# Documentation du Firmware ESP32 — Robot Quadrupède

**Projet :** QuadBot — Robot quadrupède autonome  
**Microcontrôleur :** ESP32 WROOM-32  
**Composants pilotés :** PCA9685 (I2C) · 8 servo-moteurs · Capteur MQ-9  
**Communication :** WebSocket port 81 · HTTP port 80

---

## Architecture générale

```
┌─────────────────── loop() — tourne en continu ───────────────────────┐
│                                                                       │
│  httpServer.handleClient()  ←── répond aux requêtes HTTP port 80      │
│  wsServer.loop()            ←── traite les messages WebSocket port 81 │
│                │                                                      │
│                └──► onWsEvent()  ←── parse JSON, met à jour currentCmd│
│                                                                       │
│  tickGait()   ←── avance d'une frame si le délai est écoulé          │
│       │                                                               │
│       └──► applyFrame()  ──► setServo() x8  ──► PCA9685 (I2C)        │
│                                                        │              │
│                                                   8 servo-moteurs     │
│                                                                       │
│  toutes les 2 s : readMQ9()  ──► JSON ──► wsServer.broadcastTXT()    │
└───────────────────────────────────────────────────────────────────────┘
```

### Légende des tags

| Tag | Signification |
|-----|---------------|
| `[WS]` | Échange WebSocket |
| `[PCA]` | Pilotage PCA9685 / servos via I2C |
| `[MQ9]` | Lecture capteur gaz MQ-9 |
| `[NET]` | Envoi réseau vers le dashboard |

### Variables d'état globales

```cpp
// Commande en cours — mise à jour par onWsEvent(), lue par tickGait()
enum Command { CMD_STOP, CMD_FORWARD, CMD_BACKWARD, CMD_LEFT, CMD_RIGHT };
volatile Command currentCmd = CMD_STOP;

// Vitesse 0-100 reçue du dashboard
volatile uint8_t speed = 50;

// Index de la frame de marche en cours
uint8_t  gaitFrame      = 0;

// Timestamps pour les timers non-bloquants
uint32_t lastGaitStep   = 0;  // Dernière frame de marche
uint32_t lastSensorSend = 0;  // Dernier envoi MQ-9
```

> **Pourquoi `volatile` ?**  
> `currentCmd` et `speed` sont modifiées depuis le handler WebSocket et lues depuis la boucle principale — deux contextes d'exécution différents. Le mot-clé `volatile` empêche le compilateur de les mettre en cache dans un registre, garantissant que chaque lecture accède bien à la valeur en mémoire.

---
## 1. `setup()` — Initialisation `[WS]` `[PCA]` `[NET]`

### Rôle

Exécutée **une seule fois** au démarrage. Elle initialise dans l'ordre : les pins GPIO, le PCA9685 via I2C, la connexion WiFi, puis les deux serveurs réseau (HTTP port 80 et WebSocket port 81).

### Code annoté

```cpp
void setup() {
  Serial.begin(115200);

  // ── 1. Pins MQ-9 ─────────────────────────────────────────────
  // GPIO34 = entrée analogique (ADC1, compatible WiFi actif)
  // GPIO32 = entrée digitale (seuil du comparateur interne du module)
  pinMode(MQ9_ANALOG_PIN,  INPUT);
  pinMode(MQ9_DIGITAL_PIN, INPUT);

  // ── 2. PCA9685 via I2C (SDA=GPIO21, SCL=GPIO22) ──────────────
  // Adresse I2C par défaut : 0x40 (toutes les cavalières ouvertes)
  Wire.begin();
  pca.begin();
  pca.setOscillatorFrequency(27000000); // Fréquence réelle du quartz du module
  pca.setPWMFreq(SERVO_FREQ);           // 50 Hz = fréquence standard servo
  delay(10);
  standStill(); // Place tous les servos en position neutre au démarrage

  // ── 3. Connexion WiFi ─────────────────────────────────────────
  WiFi.begin(SSID, PASSWORD);
  while (WiFi.status() != WL_CONNECTED) {
    delay(500);
    Serial.print(".");
  }
  // Affiche l'IP à entrer dans le dashboard
  Serial.printf("\nIP : http://%s\n", WiFi.localIP().toString().c_str());
  Serial.printf("WebSocket : ws://%s:81/ws\n", WiFi.localIP().toString().c_str());

  // ── 4. Serveur HTTP port 80 ───────────────────────────────────
  httpServer.on("/", handleRoot);
  httpServer.begin();

  // ── 5. Serveur WebSocket port 81 ─────────────────────────────
  // onWsEvent = callback appelé à chaque événement WebSocket
  wsServer.begin();
  wsServer.onEvent(onWsEvent);
}
```

> **Pourquoi `setOscillatorFrequency(27000000)` ?**  
> Le PCA9685 possède un oscillateur interne censé fonctionner à 25 MHz, mais la plupart des modules bon marché oscillent réellement à ~27 MHz. Sans cette correction, la fréquence PWM serait ~54 Hz au lieu de 50 Hz, et les servos recevraient des impulsions légèrement décalées — ce qui les ferait vibrer ou se positionner incorrectement.

---
## 2. `loop()` — Boucle principale `[WS]` `[PCA]` `[MQ9]`

### Rôle

Tourne en permanence après `setup()`. Elle ne contient **aucun `delay()`** — tout est géré par des timers non-bloquants basés sur `millis()`. Cela permet de traiter les WebSocket et de faire avancer la marche simultanément sans qu'une tâche bloque l'autre.

### Code annoté

```cpp
void loop() {
  // ── Tâche 1 : réseau ─────────────────────────────────────────
  // handleClient() traite les requêtes HTTP en attente
  // wsServer.loop() traite les messages WebSocket et appelle onWsEvent()
  httpServer.handleClient();
  wsServer.loop();

  // ── Tâche 2 : marche ─────────────────────────────────────────
  // tickGait() vérifie en interne si le délai entre frames est écoulé
  // Si oui → avance d'une frame et pilote les servos via PCA9685
  // Si non → retour immédiat sans rien faire (non-bloquant)
  tickGait();

  // ── Tâche 3 : envoi capteur MQ-9 toutes les 2 secondes ───────
  if (millis() - lastSensorSend > 2000) {
    lastSensorSend = millis();

    SensorData d = readMQ9();

    // Sérialise en JSON et diffuse à tous les clients WebSocket
    StaticJsonDocument<160> doc;
    doc["type"]    = "sensor";
    doc["raw"]     = d.raw;
    doc["voltage"] = serialized(String(d.voltage, 3));
    doc["rs"]      = serialized(String(d.rs,      3));
    doc["ratio"]   = serialized(String(d.ratio,   3));
    doc["ppm"]     = serialized(String(d.ppm,     1));
    doc["alert"]   = d.alert;

    String out;
    serializeJson(doc, out);
    wsServer.broadcastTXT(out);
  }
}
```

> **Pourquoi pas de `delay()` ?**  
> Un `delay(100)` bloquerait tout le firmware pendant 100 ms : aucun paquet WebSocket ne serait traité, la marche serait hachée, et le dashboard verrait la connexion comme instable. La technique `millis()` permet de vérifier si le temps est écoulé sans jamais bloquer le processeur — les trois tâches tournent en quasi-simultané.

---
## 3. `onWsEvent()` — Réception des commandes `[WS]`

### Rôle

Fonction callback appelée automatiquement par la librairie `WebSocketsServer` à chaque événement réseau. Elle reçoit les messages JSON du dashboard, les décode avec ArduinoJson, et met à jour `currentCmd` et `speed` — les deux variables que `tickGait()` lit pour savoir quoi faire.

### Paramètres

| Paramètre | Type | Description |
|-----------|------|-------------|
| `num` | `uint8_t` | Identifiant du client WebSocket (0, 1, 2...) |
| `type` | `WStype_t` | Type d'événement : `WStype_TEXT`, `WStype_DISCONNECTED`, etc. |
| `payload` | `uint8_t*` | Contenu brut du message reçu |
| `length` | `size_t` | Longueur du payload en octets |

### Code annoté

```cpp
void onWsEvent(uint8_t num, WStype_t type,
               uint8_t* payload, size_t length) {

  // WStype_TEXT = message texte reçu (le dashboard envoie toujours du JSON)
  if (type == WStype_TEXT) {

    // Décode le JSON dans un buffer de 128 octets alloué sur la pile
    // StaticJsonDocument n'utilise pas malloc → plus sûr sur microcontrôleur
    StaticJsonDocument<128> doc;
    if (deserializeJson(doc, payload, length)) return; // JSON invalide → ignore

    const char* t = doc["type"];
    if (!t) return;

    // ── Ping → répondre immédiatement avec un pong ────────────────
    // Le dashboard mesure (Date.now() - pingTime) pour calculer la latence
    if (strcmp(t, "ping") == 0) {
      wsServer.sendTXT(num, "{\"type\":\"pong\"}");
      return;
    }

    // ── Commande de déplacement ───────────────────────────────────
    if (strcmp(t, "cmd") == 0) {
      const char* cmd = doc["cmd"];

      // Extrait la vitesse (0-100), valeur par défaut 50 si absente
      speed = constrain((int)doc["speed"] | 50, 0, 100);

      // Met à jour currentCmd — tickGait() la lira au prochain cycle
      if      (strcmp(cmd, "forward")  == 0) currentCmd = CMD_FORWARD;
      else if (strcmp(cmd, "backward") == 0) currentCmd = CMD_BACKWARD;
      else if (strcmp(cmd, "left")     == 0) currentCmd = CMD_LEFT;
      else if (strcmp(cmd, "right")    == 0) currentCmd = CMD_RIGHT;
      else                                    currentCmd = CMD_STOP;

      gaitFrame = 0; // Repart du début de la séquence à chaque nouvelle commande
    }
  }

  // ── Déconnexion d'un client → sécurité : immobiliser le robot ──
  // Si le WiFi coupe ou le navigateur est fermé, le robot s'arrête
  if (type == WStype_DISCONNECTED) {
    currentCmd = CMD_STOP;
    standStill(); // Tous les servos en position neutre immédiatement
  }
}
```

> **`sendTXT` vs `broadcastTXT`**  
> `wsServer.sendTXT(num, msg)` envoie uniquement au client identifié par `num` (celui qui a envoyé le ping).  
> `wsServer.broadcastTXT(msg)` envoie à **tous** les clients connectés simultanément — utilisé pour les données MQ-9.

> **Sécurité déconnexion**  
> Sans le `standStill()` dans `WStype_DISCONNECTED`, les servos continueraient la dernière frame de marche indéfiniment si le WiFi coupe — le robot marcherait sans contrôle.

---
## 4. `tickGait()` — Moteur de marche `[PCA]`

### Rôle

Appelée à chaque tour de `loop()`, elle décide si le moment est venu d'avancer d'une frame de marche. Elle utilise un timer non-bloquant basé sur `millis()` et sélectionne la bonne séquence selon `currentCmd`.

### Code annoté

```cpp
// Calcule le délai entre deux frames selon la vitesse reçue du dashboard
// speed=0   → 400 ms entre chaque frame (très lent)
// speed=100 → 80 ms entre chaque frame  (rapide)
uint32_t gaitDelay() {
  return (uint32_t)map(speed, 0, 100, 400, 80);
}

void tickGait() {
  // Timer non-bloquant : si le délai n'est pas écoulé, retour immédiat
  if (millis() - lastGaitStep < gaitDelay()) return;
  lastGaitStep = millis();

  // Sélectionne la séquence de marche selon la commande active
  switch (currentCmd) {

    case CMD_FORWARD:
      // GAIT_FORWARD contient 4 frames → modulo 4 pour boucler indéfiniment
      applyFrame(GAIT_FORWARD[gaitFrame % 4]);
      gaitFrame++;
      break;

    case CMD_BACKWARD:
      applyFrame(GAIT_BACKWARD[gaitFrame % 4]);
      gaitFrame++;
      break;

    case CMD_LEFT:
      // Les virages n'ont que 2 frames
      applyFrame(GAIT_LEFT[gaitFrame % 2]);
      gaitFrame++;
      break;

    case CMD_RIGHT:
      applyFrame(GAIT_RIGHT[gaitFrame % 2]);
      gaitFrame++;
      break;

    case CMD_STOP:
    default:
      standStill();
      break;
  }
}
```

### Relation vitesse ↔ délai

| `speed` (dashboard) | `gaitDelay()` | Cadence |
|---------------------|---------------|----------|
| 0 | 400 ms | ~2.5 frames/s — très lent |
| 25 | 320 ms | ~3 frames/s |
| 50 | 240 ms | ~4 frames/s — vitesse normale |
| 75 | 160 ms | ~6 frames/s |
| 100 | 80 ms | ~12 frames/s — rapide |

---
## 5. Séquences de marche — `GaitFrame` `[PCA]`

### Structure

Chaque `GaitFrame` est un **instantané de position** : il décrit exactement où doit être chaque hanche et chaque genou des 4 membres à un instant donné. Enchaîner plusieurs frames crée l'illusion du mouvement.

```cpp
// Position d'un membre : hanche + genou, en ticks PCA9685
struct LegPos { uint16_t hip; uint16_t knee; };

// Un instantané complet des 4 membres
struct GaitFrame {
  LegPos br;  // Postérieur Droit  → ch0 (LED01) + ch1 (LED02)
  LegPos bl;  // Postérieur Gauche → ch2 (LED03) + ch3 (LED04)
  LegPos fr;  // Antérieur Droit   → ch4 (LED05) + ch5 (LED06)
  LegPos fl;  // Antérieur Gauche  → ch6 (LED07) + ch7 (LED08)
};

// Ticks PCA9685 à 50 Hz (4096 ticks = une période de 20 ms)
#define SERVO_MIN  150    // ~0°
#define SERVO_MID  375    // ~90° (position neutre)
#define SERVO_MAX  600    // ~180°
```

### Principe du trot diagonal

La marche utilise un **trot diagonal** : deux membres diagonalement opposés bougent ensemble pendant que les deux autres restent en appui. C'est la démarche naturelle des quadrupèdes à vitesse moyenne.

```
Phase 1 : FR + BL lèvent         Phase 2 : FL + BR lèvent
┌────────────────────┐            ┌────────────────────┐
│  FL (appui)  FR ↑  │            │  FL ↑   FR (appui) │
│       CORPS        │            │       CORPS        │
│  BL ↑   BR (appui) │            │  BL (appui)  BR ↑  │
└────────────────────┘            └────────────────────┘
↑ = pied levé (avance)            (appui) = pied au sol
```

### Séquence GAIT_FORWARD — 4 frames

```cpp
const GaitFrame GAIT_FORWARD[4] = {

  // Frame 0 : FR et BL se lèvent — FL et BR restent en appui
  { {SERVO_MID,      SERVO_MID},      // BR : hanche et genou neutres (appui)
    {SERVO_MID+50,   SERVO_MID-60},   // BL : hanche avance, genou se lève
    {SERVO_MID-50,   SERVO_MID-60},   // FR : hanche avance, genou se lève
    {SERVO_MID,      SERVO_MID} },    // FL : neutre (appui)

  // Frame 1 : FR et BL se posent en avant
  { {SERVO_MID,      SERVO_MID},
    {SERVO_MID-30,   SERVO_MID},      // BL : genou redescend, pied se pose
    {SERVO_MID+30,   SERVO_MID},      // FR : genou redescend, pied se pose
    {SERVO_MID,      SERVO_MID} },

  // Frame 2 : FL et BR se lèvent — FR et BL restent en appui
  { {SERVO_MID+50,   SERVO_MID-60},   // BR : hanche avance, genou se lève
    {SERVO_MID,      SERVO_MID},
    {SERVO_MID,      SERVO_MID},
    {SERVO_MID-50,   SERVO_MID-60} }, // FL : hanche avance, genou se lève

  // Frame 3 : FL et BR se posent en avant
  { {SERVO_MID-30,   SERVO_MID},
    {SERVO_MID,      SERVO_MID},
    {SERVO_MID,      SERVO_MID},
    {SERVO_MID+30,   SERVO_MID} }
};
```

> **Calibration des frames**  
> Les deltas comme `+50` ou `-60` sont des points de départ. Pour affiner : branche un seul servo, envoie manuellement des ticks via `pca.setPWM(ch, 0, valeur)` dans le moniteur série, et observe l'angle obtenu. Ajuste ensuite les GaitFrames selon la géométrie réelle de tes membres.

---
## 6. `applyFrame()` — Application d'une frame `[PCA]`

### Rôle

Reçoit un `GaitFrame` et envoie les 8 positions (4 hanches + 4 genoux) au PCA9685 via I2C. C'est la fonction qui **traduit les données en mouvement physique**.

### Code annoté

```cpp
void applyFrame(const GaitFrame& f) {
  // Postérieur Droit  — LED01 (ch0) = hanche, LED02 (ch1) = genou
  setServo(SERVO_BR_HIP,  f.br.hip);
  setServo(SERVO_BR_KNEE, f.br.knee);

  // Postérieur Gauche — LED03 (ch2) = hanche, LED04 (ch3) = genou
  setServo(SERVO_BL_HIP,  f.bl.hip);
  setServo(SERVO_BL_KNEE, f.bl.knee);

  // Antérieur Droit   — LED05 (ch4) = hanche, LED06 (ch5) = genou
  setServo(SERVO_FR_HIP,  f.fr.hip);
  setServo(SERVO_FR_KNEE, f.fr.knee);

  // Antérieur Gauche  — LED07 (ch6) = hanche, LED08 (ch7) = genou
  setServo(SERVO_FL_HIP,  f.fl.hip);
  setServo(SERVO_FL_KNEE, f.fl.knee);
}
```

### Mapping complet des canaux PCA9685

| Constante C++ | Canal PCA | LED module | Membre | Articulation |
|---------------|-----------|-----------|--------|-------------|
| `SERVO_BR_HIP` | ch 0 | LED01 | Postérieur Droit | Hanche |
| `SERVO_BR_KNEE` | ch 1 | LED02 | Postérieur Droit | Genou |
| `SERVO_BL_HIP` | ch 2 | LED03 | Postérieur Gauche | Hanche |
| `SERVO_BL_KNEE` | ch 3 | LED04 | Postérieur Gauche | Genou |
| `SERVO_FR_HIP` | ch 4 | LED05 | Antérieur Droit | Hanche |
| `SERVO_FR_KNEE` | ch 5 | LED06 | Antérieur Droit | Genou |
| `SERVO_FL_HIP` | ch 6 | LED07 | Antérieur Gauche | Hanche |
| `SERVO_FL_KNEE` | ch 7 | LED08 | Antérieur Gauche | Genou |

---
## 7. `setServo()` — Pilotage d'un servo `[PCA]`

### Rôle

Envoie une commande PWM à un canal précis du PCA9685. Elle **borne la valeur** entre `SERVO_MIN` et `SERVO_MAX` avant l'envoi — un dépassement mécaniquement dangereux est ainsi impossible.

### Code annoté

```cpp
void setServo(uint8_t ch, uint16_t ticks) {
  // constrain() empêche d'envoyer une valeur hors des limites mécaniques
  // Un servo forcé hors de sa plage peut brûler son moteur interne
  ticks = constrain(ticks, SERVO_MIN, SERVO_MAX);

  // pca.setPWM(canal, délai_début, délai_fin)
  //   canal       = 0 à 15 sur le PCA9685
  //   délai_début = 0  (impulsion commence au début de la période 20 ms)
  //   délai_fin   = ticks (nombre de ticks avant la fin de l'impulsion)
  //
  // À 50 Hz : période = 20 ms = 4096 ticks
  //   150 ticks → 150/4096 × 20 ms ≈ 0.73 ms → ~0°
  //   375 ticks → 375/4096 × 20 ms ≈ 1.83 ms → ~90°
  //   600 ticks → 600/4096 × 20 ms ≈ 2.93 ms → ~180°
  pca.setPWM(ch, 0, ticks);
}
```

### Correspondance ticks ↔ angle

| Ticks | Durée impulsion | Angle approximatif | Rôle |
|-------|----------------|--------------------|------|
| `150` (SERVO_MIN) | ~0.73 ms | ~0° | Limite basse |
| `375` (SERVO_MID) | ~1.83 ms | ~90° | Position neutre |
| `600` (SERVO_MAX) | ~2.93 ms | ~180° | Limite haute |

> **Ces valeurs sont à calibrer** selon ton modèle de servo. La plage standard est 1 ms–2 ms, mais beaucoup de servos acceptent 0.5 ms–2.5 ms. Ajuste `SERVO_MIN` et `SERVO_MAX` en conséquence.

---
## 8. `standStill()` — Posture neutre `[PCA]`

### Rôle

Place tous les servos en position `SERVO_MID` (90°). Appelée au **démarrage**, à l'**arrêt** et à la **déconnexion** du dashboard. C'est la posture de repos du robot — debout, stable, prêt à recevoir des commandes.

### Code annoté

```cpp
const LegPos STAND = { SERVO_MID, SERVO_MID }; // { 375, 375 }

void standStill() {
  // Construit un GaitFrame où tous les membres sont à 90°
  // La syntaxe { STAND, STAND, STAND, STAND } initialise
  // les 4 champs br, bl, fr, fl avec la même valeur STAND
  applyFrame({ STAND, STAND, STAND, STAND });

  // Remet le compteur de frame à 0 pour que la prochaine commande
  // reparte proprement depuis le début de sa séquence
  gaitFrame = 0;
}
```

### Quand est-elle appelée ?

| Moment | Raison |
|--------|--------|
| `setup()` | Positionne les servos dès le démarrage, avant toute commande |
| `onWsEvent()` → `WStype_DISCONNECTED` | Sécurité si le dashboard se déconnecte |
| `tickGait()` → `CMD_STOP` | Arrêt normal sur commande `"stop"` |

---
## 9. `readMQ9()` — Lecture du capteur gaz `[MQ9]`

### Rôle

Lit les deux sorties du module MQ-9 et calcule une estimation de concentration CO en ppm. Le calcul suit la courbe caractéristique de la datasheet MQ-9, modélisée par une loi puissance log-log.

### Principe électronique du MQ-9

```
VCC (3.3V)
    │
   RS  ← résistance interne du capteur (varie avec le gaz)
    │
   Vout ──────────────────────► GPIO34 (ADC)
    │
   RL  ← résistance de charge (10 kΩ sur le module)
    │
  GND

D'où : RS = RL × (VCC - Vout) / Vout
```

Quand la concentration de gaz augmente, RS **diminue** → Vout **augmente** → ratio RS/RO **diminue** → ppm **augmente**.

### Code annoté

```cpp
SensorData readMQ9() {
  SensorData d;

  // ── Lecture analogique (GPIO34, ADC1) ─────────────────────────
  // analogRead() retourne 0-4095 (résolution 12 bits de l'ESP32)
  // GPIO34 est sur ADC1 → compatible avec le WiFi actif
  // (ADC2 est partagé avec le WiFi et retourne 0 quand WiFi est actif)
  d.raw = analogRead(MQ9_ANALOG_PIN);

  // ── Lecture digitale (GPIO32) ─────────────────────────────────
  // LOW = le comparateur interne du module a détecté un dépassement
  // Le seuil est réglable via le potentiomètre bleu du module
  d.alert = digitalRead(MQ9_DIGITAL_PIN) == LOW;

  // ── Conversion en volts ───────────────────────────────────────
  // L'ADC 12 bits de l'ESP32 couvre 0 à 3.3V
  // tension = (raw / 4095) × 3.3
  d.voltage = (d.raw / 4095.0f) * 3.3f;

  // ── Calcul de RS (résistance du capteur) ──────────────────────
  // Pont diviseur de tension : RS = RL × (VCC - Vout) / Vout
  // RL_VALUE = 10 kΩ (valeur typique sur les modules MQ-9)
  if (d.voltage > 0.01f) {
    d.rs = RL_VALUE * (3.3f - d.voltage) / d.voltage;
  } else {
    d.rs = 0;
  }

  // ── Calcul du ratio RS/RO ─────────────────────────────────────
  // RO = résistance dans l'air pur (à calibrer — voir note ci-dessous)
  // Plus le ratio est bas, plus la concentration de gaz est élevée
  d.ratio = d.rs / RO;

  // ── Estimation CO en ppm (courbe datasheet MQ-9) ─────────────
  // La datasheet donne une droite en échelle log-log :
  //   ppm = a × (RS/RO)^b
  // Pour le CO : a = 599.65, b = -2.244
  // (ajustés depuis la courbe graphique de la datasheet)
  if (d.ratio > 0) {
    d.ppm = 599.65f * pow(d.ratio, -2.244f);
  } else {
    d.ppm = 0;
  }

  return d;
}
```

### Description des champs retournés

| Champ | Unité | Description |
|-------|-------|-------------|
| `raw` | 0 – 4095 | Valeur brute ADC 12 bits |
| `voltage` | V | Tension mesurée en sortie du capteur |
| `rs` | kΩ | Résistance interne RS du capteur |
| `ratio` | — | RS/RO — entrée de la courbe datasheet |
| `ppm` | ppm | Estimation CO par la loi puissance |
| `alert` | bool | `true` si seuil digital (GPIO32) dépassé |

> **Calibration obligatoire — la valeur `RO`**  
> La constante `RO` (résistance dans l'air pur) varie d'un capteur à l'autre. Pour calibrer :  
> 1. Laisse le MQ-9 chauffer **5 minutes en air pur**  
> 2. Lis la valeur de `d.rs` dans le moniteur série  
> 3. Remplace `#define RO 10.0` par cette valeur  
> Sans calibration, les ppm affichées sont indicatives mais pas précises.

---
## 10. Envoi des données au dashboard — `broadcastTXT()` `[NET]`

### Rôle

Toutes les 2 secondes, l'ESP sérialise les données du MQ-9 en JSON et les diffuse à **tous les clients WebSocket connectés** simultanément. Le dashboard reçoit ce JSON dans `ws.onmessage` et appelle `updateSensors()`.

### Code annoté

```cpp
if (millis() - lastSensorSend > 2000) {
  lastSensorSend = millis();

  SensorData d = readMQ9();

  // StaticJsonDocument<160> = buffer JSON de 160 octets sur la pile
  // Taille calculée : ~120 octets pour ce JSON + marge de sécurité
  StaticJsonDocument<160> doc;

  doc["type"]    = "sensor";
  doc["raw"]     = d.raw;

  // serialized(String(valeur, décimales)) insère un float pré-formaté
  // sans laisser ArduinoJson choisir le nombre de décimales
  doc["voltage"] = serialized(String(d.voltage, 3)); // ex: "1.482"
  doc["rs"]      = serialized(String(d.rs,      3)); // ex: "12.340"
  doc["ratio"]   = serialized(String(d.ratio,   3)); // ex: "1.234"
  doc["ppm"]     = serialized(String(d.ppm,     1)); // ex: "47.3"
  doc["alert"]   = d.alert;                          // true / false

  String out;
  serializeJson(doc, out); // Convertit le document en String JSON

  // broadcastTXT() envoie à TOUS les clients connectés
  // sendTXT(num, ...) enverrait uniquement au client numéro "num"
  wsServer.broadcastTXT(out);

  // Log série pour le moniteur PlatformIO
  Serial.printf("[MQ-9] %.1f ppm | alert=%s | CMD=%d speed=%d\n",
    d.ppm, d.alert ? "OUI" : "non", (int)currentCmd, speed);
}
```

### JSON produit et envoyé

```json
{
  "type":    "sensor",
  "raw":     1842,
  "voltage": "1.482",
  "rs":      "12.340",
  "ratio":   "1.234",
  "ppm":     "47.3",
  "alert":   false
}
```

---
## 11. Flux complet — du clic au servo

```
DASHBOARD (navigateur)                     ESP32 (firmware C++)

Appui flèche ↑
  startCmd('forward')
  sendCmd('forward')
    ws.send({"type":"cmd",           ──►  onWsEvent()
             "cmd":"forward",               deserializeJson()
             "speed":50})                   currentCmd = CMD_FORWARD
                                            gaitFrame  = 0

  [setInterval 150ms répète sendCmd]  ──►  onWsEvent() [ignoré si même cmd]

                                       loop() → tickGait()
                                         millis() - lastGaitStep >= gaitDelay()
                                         applyFrame(GAIT_FORWARD[0])
                                           setServo(ch0, 375) ──► PCA9685 I2C
                                           setServo(ch1, 315) ──► PCA9685 I2C
                                           setServo(ch2, 425) ──► PCA9685 I2C
                                           ...× 8 servos
                                                    │
                                              servos bougent

  [toutes les 2 s, en parallèle]       loop() → readMQ9()
                                          analogRead(GPIO34)
                                          calcul RS → ratio → ppm
                                          broadcastTXT(JSON sensor)
    ws.onmessage → updateSensors() ◄──
      ppm, barre, alertes mis à jour

Relâche flèche ↑
  stopCmd()
  sendCmd('stop')
    ws.send({"type":"cmd",           ──►  onWsEvent()
             "cmd":"stop"})                 currentCmd = CMD_STOP

                                       tickGait() → standStill()
                                         applyFrame(STAND × 4)
                                           tous servos → 375 ticks (90°)
```

---

## Résumé des fonctions

| Fonction | Tag | Rôle résumé |
|----------|-----|-------------|
| `setup()` | `[WS]` `[PCA]` | Initialise GPIO, PCA9685, WiFi, HTTP et WebSocket |
| `loop()` | `[WS]` `[PCA]` `[MQ9]` | Boucle principale — 3 tâches non-bloquantes |
| `onWsEvent()` | `[WS]` | Reçoit et route les JSON du dashboard |
| `tickGait()` | `[PCA]` | Timer de marche — avance d'une frame selon la vitesse |
| `applyFrame()` | `[PCA]` | Envoie 8 positions servo au PCA9685 via I2C |
| `setServo()` | `[PCA]` | Envoie un tick PWM sécurisé à un canal PCA9685 |
| `standStill()` | `[PCA]` | Place tous les servos en position neutre 90° |
| `readMQ9()` | `[MQ9]` | Lit l'ADC et calcule tension, RS, ratio, ppm, alerte |
| `broadcastTXT()` | `[NET]` | Sérialise et diffuse le JSON capteur à tous les clients |